In [ ]:
pip install pandas pyarrow

In [ ]:
pip install endee

In [ ]:
pip show endee

In [2]:
base_path = "/home/debian/latest_VDB/VectorDBBench/vectordataset_label"
dataset_folder = "cohere/cohere_medium_1m"

In [ ]:
#For checking parquet file contents
import pyarrow.parquet as pq
import os

file_name = "shuffle_train.parquet"  #input

# Build full path
file_path = os.path.join(base_path, dataset_folder, file_name)


parquet_file = pq.ParquetFile(file_path)

# Read only first batch of rows
first_batch = next(parquet_file.iter_batches(batch_size=5))
preview = first_batch.to_pandas()

for col in preview.columns:
    if preview[col].dtype == object and isinstance(preview[col].iloc[0], list):
        preview[col] = preview[col].apply(lambda x: x[:5] if x is not None else x)

print(preview)

In [4]:
from endee import Endee
client = Endee(token="localtest")
client.set_base_url("http://148.113.54.173:8080/api/v1")

In [ ]:
#For checking indexes
for obj in client.list_indexes().get('indexes'):
    print(obj['name'])
    print(obj['total_elements'])
    print('\t')

In [6]:
#give the index name
index_name = "test_1M_labelfilter_int16_latest_master"
index = client.get_index(index_name)

In [ ]:
index.describe()

In [ ]:
#For querying
import pyarrow.parquet as pq
import os

#inputs
label_to_query = "label_1p"
top_k = 30

# Mode 1: Single query id
# Mode 2: Find all query ids with recall < 1
mode = 2          # change to 1 or 2
query_id = 457    # only used in mode 1

# Build full paths
dataset_path = os.path.join(base_path, dataset_folder)
test_parquet_path = os.path.join(dataset_path, "test.parquet")
gt_parquet_path = os.path.join(dataset_path, f"neighbors_labels_{label_to_query}.parquet")

# Read parquet files
test_df = pq.ParquetFile(test_parquet_path).read().to_pandas()
gt_df = pq.ParquetFile(gt_parquet_path).read().to_pandas()

def run_query(query_id):
    query_row = test_df[test_df["id"] == query_id]
    if query_row.empty:
        print(f"Query ID {query_id} not found in test.parquet")
        return None

    query_vector = query_row["emb"].values[0]

    gt_row = gt_df[gt_df["id"] == query_id]
    if gt_row.empty:
        print(f"Query ID {query_id} not found in ground truth file")
        return None

    ground_truth = gt_row["neighbors_id"].values[0][:top_k]

    results = index.query(
        vector=query_vector,
        top_k=top_k,
        filter=[{"label": {"$eq": label_to_query}}],
        include_vectors=False,
    )
    returned_ids = [int(r["id"]) for r in results]

    intersection = len(set(returned_ids) & set(ground_truth))
    recall = intersection / len(ground_truth)

    return returned_ids, ground_truth, recall


if mode == 1:
    result = run_query(query_id)
    if result:
        returned_ids, ground_truth, recall = result
        print(f"Query ID: {query_id}")
        print(f"Returned IDs: {','.join(map(str, returned_ids))}")
        print(f"Ground Truth: {','.join(map(str, ground_truth))}")
        print(f"Recall: {recall:.4f}")

elif mode == 2:
    print(f"Finding all query IDs with recall < 1.0 ...\n")
    low_recall_ids = []
    all_recalls = []
    for query_id in test_df["id"].values:
        result = run_query(query_id)
        if result:
            returned_ids, ground_truth, recall = result
            all_recalls.append(recall)
            if recall < 1.0:
                low_recall_ids.append(query_id)
                print(f"Query ID: {query_id}")
                print(f"Returned IDs: {','.join(map(str, returned_ids))}")
                print(f"Ground Truth: {','.join(map(str, ground_truth))}")
                print(f"Recall: {recall:.4f}\n")

    print(f"Total query IDs with recall < 1.0: {len(low_recall_ids)}")
    print(f"IDs: {low_recall_ids}")
    print(f"Average Recall across all {len(all_recalls)} queries: {sum(all_recalls)/len(all_recalls):.4f}")

In [ ]:
index.describe()

In [ ]:
#For deleting
# Delete by label
label_to_delete = "label_1p" #input

result = index.delete_with_filter([{"label": {"$eq": label_to_delete}}])
print(f"Deleted vectors with label: {label_to_delete}")
print(result)

In [ ]:
index.describe()

In [ ]:
#For reinserting delted vectors
import pyarrow.parquet as pq
import pandas as pd
import os

# User inputs
label_to_insert = "label_1p"
batch_size = 1000

# Build full paths
dataset_path = os.path.join(base_path, dataset_folder)
labels_path = os.path.join(dataset_path, "scalar_labels.parquet")
emb_path = os.path.join(dataset_path, "shuffle_train.parquet")

# Read labels file fully (small file, just id + label)
labels_df = pq.ParquetFile(labels_path).read().to_pandas()

# Filter only the label we need
filtered_labels = labels_df[labels_df["labels"] == label_to_insert]
valid_ids = set(filtered_labels["id"].values)

print(f"Total vectors to insert: {len(valid_ids)}")

# Process shuffle_train in batches to avoid RAM issues
emb_file = pq.ParquetFile(emb_path)
total_inserted = 0

for batch in emb_file.iter_batches(batch_size=batch_size):
    batch_df = batch.to_pandas()
    
    # Keep only rows whose id is in filtered labels
    batch_df = batch_df[batch_df["id"].isin(valid_ids)]
    
    if batch_df.empty:
        continue
    
    # Merge to get labels
    batch_df = batch_df.merge(filtered_labels[["id", "labels"]], on="id")
    
    batch_vectors = []
    for _, row in batch_df.iterrows():
        vector_data = {
            "id": str(row["id"]),
            "vector": row["emb"],
            "meta": {"id": row["id"]},
            "filter": {
                "id": row["id"],
                "label": row["labels"]
            }
        }
        batch_vectors.append(vector_data)
    
    index.upsert(batch_vectors)
    total_inserted += len(batch_vectors)
    print(f"Upserted {len(batch_vectors)} vectors, total so far: {total_inserted}")

print(f"Done! Total inserted: {total_inserted} vectors with label '{label_to_insert}'")

In [ ]:
index.describe()